# 2026-03 Basics of tesseract

- Example and fake data used for this study:
    - Microsoft Azure invoices: https://www.microsoft.com/en-us/download/details.aspx?id=38805
    - Kaggle receipts (2024-EN): https://www.kaggle.com/datasets/jenswalter/receipts

**Note**: Tesseract does not read PDFs directly. The typical workflow is to first render each PDF page as an image (using `pymupdf` or `pdf2image`) and then pass it to `pytesseract`. Higher render resolution (zoom ≥ 2×) leads to better OCR accuracy.

---

### Setup
Importing all necessary libs and getting a list with all the example data

In [11]:
# Python imports
import os, sys
from pathlib import Path

# Data handling imports
import pandas as pd
import numpy as np
from tqdm import tqdm
from PIL import Image

# --> Main imports: pytesseract + pymupdf (to render PDF pages as images)
import pytesseract
from pytesseract import Output
import pymupdf

In [12]:
# Path to all the example files
path = Path('./data/')

# Get all files
files_list = [f'./data/{entry.name}' for entry in path.rglob('*') if entry.is_file()]

# Sort the list
files_list = sorted(files_list)

files_list

['./data/example-invoice-microsoft-azure-PAYG.pdf',
 './data/example-receipts-kaggle-cheesecakefactory.pdf',
 './data/example-receipts-kaggle-luckylouie.pdf',
 './data/example-receipts-kaggle-mcdonald.pdf',
 './data/example-receipts-kaggle-oldnavy.pdf',
 './data/example-receipts-kaggle-spaceneedle.pdf',
 './data/example-receipts-kaggle-wholefoods_1.pdf',
 './data/example-receipts-kaggle-wholefoods_2.pdf']

### Helper functions

In [13]:
def helper_pdf_page_to_image(pdf_path: str, page_number: int = 0, zoom: float = 2.0):
    """
    Convert a PDF page to a PIL Image ready for pytesseract.
    Higher zoom → higher DPI → better OCR accuracy (zoom=2.0 gives ~144 DPI).
    """
    doc = pymupdf.open(pdf_path)
    page = doc.load_page(page_number)

    # Render to pixmap at the requested zoom level
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    # Convert pymupdf pixmap to PIL Image
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    print("="*80)
    print(f"PDF: {pdf_path} | Page: {page_number}")
    print(f"Image size: {img.width}x{img.height} px | Mode: {img.mode} | Zoom: {zoom}x")
    print("="*80)

    return img

In [14]:
def helper_ocr_report(text: str, source: str = ""):
    """
    Helper function to display a summary of OCR-extracted text.
    """
    lines = [l for l in text.splitlines() if l.strip()]
    words = text.split()

    print("="*80)
    if source:
        print(f"Source: {source}")
    print(f"Lines: {len(lines)} | Words: {len(words)} | Chars: {len(text)}")
    print("="*80)
    print(text[:500])
    print("")

### Basic testing
Just simple 'out of the box' examples and pipelines

In [15]:
# Convert the cheesecakefactory receipt PDF page to a PIL image
img = helper_pdf_page_to_image(files_list[1], page_number=0, zoom=2.0)

# image_to_string — the most direct OCR call, returns plain text
text = pytesseract.image_to_string(img)

helper_ocr_report(text, source=files_list[1])

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x
Source: ./data/example-receipts-kaggle-cheesecakefactory.pdf
Lines: 34 | Words: 142 | Chars: 790
THE CHEESECAKE FACTORY
SEATTLE

O235 TABLE 203 #Party 1
DALTON L = SvrCk: 5 17:52 05/20/24
BAR DINING

1 HH Mac & Jacks Or 5.50
1 Louisiana Chicken Pasta 25.95
1 Mac & Jacks African Amb 9.00
1 Fresh Strawberry CC 12.95

Sub Total: 53.40
Tax: 5.53
05/20 18:40 TOTAL : 58.93

Gratuity Not Included

Suggested Gratuity:
22% TIP 12.96
20% TIP 11.79
18% T1P 10.61

Use your phone's camera or visit
https://scanQR. io to scan
the code below and pay your check

We'd love Lo hear about your visit!
wow. cofs



In [16]:
# Available tesseract languages installed on the system
print("=== Tesseract version ===")
print(pytesseract.get_tesseract_version())
print()
print("=== Available languages ===")
print(pytesseract.get_languages(config=''))

=== Tesseract version ===
5.5.2

=== Available languages ===
['eng', 'osd', 'snum']


### Text Extraction
Different output modes of `pytesseract`: plain string, raw TSV data, and character bounding boxes

In [17]:
# image_to_string — plain OCR text, simplest output mode
img = helper_pdf_page_to_image(files_list[1], page_number=0, zoom=2.0)

text = pytesseract.image_to_string(img)

print("=== image_to_string ===")
print(text[:600])

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x
=== image_to_string ===
THE CHEESECAKE FACTORY
SEATTLE

O235 TABLE 203 #Party 1
DALTON L = SvrCk: 5 17:52 05/20/24
BAR DINING

1 HH Mac & Jacks Or 5.50
1 Louisiana Chicken Pasta 25.95
1 Mac & Jacks African Amb 9.00
1 Fresh Strawberry CC 12.95

Sub Total: 53.40
Tax: 5.53
05/20 18:40 TOTAL : 58.93

Gratuity Not Included

Suggested Gratuity:
22% TIP 12.96
20% TIP 11.79
18% T1P 10.61

Use your phone's camera or visit
https://scanQR. io to scan
the code below and pay your check

We'd love Lo hear about your visit!
wow. cofsurvey -com
Enter this code within 5 days:
3043-00052 -24052

Join us for Brunch, Sat/Sun until 2PM




In [18]:
# image_to_data — raw TSV output with word-level metadata and confidence scores
# Columns: level, page_num, block_num, par_num, line_num, word_num,
#          left, top, width, height, conf, text
img = helper_pdf_page_to_image(files_list[1], page_number=0, zoom=2.0)

data_tsv = pytesseract.image_to_data(img)

print("=== image_to_data (raw TSV, first 600 chars) ===\n")
print(data_tsv[:600])

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x
=== image_to_data (raw TSV, first 600 chars) ===

level	page_num	block_num	par_num	line_num	word_num	left	top	width	height	conf	text
1	1	0	0	0	0	0	0	449	1337	-1	
2	1	1	0	0	0	114	108	208	40	-1	
3	1	1	1	0	0	114	108	208	40	-1	
4	1	1	1	1	0	114	108	208	17	-1	
5	1	1	1	1	1	114	109	26	16	96.247269	THE
5	1	1	1	1	2	151	108	94	17	96.597801	CHEESECAKE
5	1	1	1	1	3	257	108	65	17	96.711937	FACTORY
4	1	1	1	2	0	189	131	65	17	-1	
5	1	1	1	2	1	189	131	65	17	96.473450	SEATTLE
2	1	2	0	0	0	26	178	344	64	-1	
3	1	2	1	0	0	26	178	344	64	-1	
4	1	2	1	1	0	27	178	278	19	-1	
5	1	2	1	1	1	27	178	73	17	86.927582	O235
5	1	2	1	1	2	122	178	47	17	95.237007	TABLE
5	1	2	1	1	3	180	17


In [19]:
# image_to_boxes — character-level bounding boxes in Tesseract coordinate system
# Format per line: char left bottom right top page
# Note: Tesseract uses bottom-left origin (y increases upward)
img = helper_pdf_page_to_image(files_list[1], page_number=0, zoom=2.0)

boxes = pytesseract.image_to_boxes(img)

print("=== image_to_boxes (first 20 character entries) ===\n")
print(f"{'char':>5}  {'left':>6}  {'bottom':>6}  {'right':>6}  {'top':>6}  {'page':>4}")
print("-" * 45)
for line in boxes.splitlines()[:20]:
    parts = line.split()
    print(f"  {parts[0]:>3}  {parts[1]:>6}  {parts[2]:>6}  {parts[3]:>6}  {parts[4]:>6}  {parts[5]:>4}")

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x
=== image_to_boxes (first 20 character entries) ===

 char    left  bottom   right     top  page
---------------------------------------------
    T     114    1213     118    1228     0
    H     122    1212     130    1228     0
    E     132    1212     140    1228     0
    C     151    1213     159    1229     0
    H     161    1213     169    1227     0
    E     171    1213     178    1229     0
    E     180    1213     188    1229     0
    S     189    1213     197    1229     0
    E     200    1213     207    1228     0
    C     208    1213     217    1228     0
    A     218    1213     226    1228     0
    K     227    1212     236    1228     0
    E     238    1212     245    1229     0
    F     257    1212     262    1228     0
    A     266    1212     274    1228     0
    C     275    1212     283    1228     0
    T     286    1212     292    122

### OCR Configuration
Controlling tesseract behaviour via `--psm` (page segmentation) and `--oem` (engine mode) flags

In [20]:
# Page Segmentation Mode (--psm N) tells tesseract how to interpret the page layout
# Common modes:
#   3  — Fully automatic page segmentation (default)
#   6  — Assume a single uniform block of text
#   11 — Sparse text: find as much text as possible in no particular order
#   12 — Sparse text with OSD
img_invoice = helper_pdf_page_to_image(files_list[0], page_number=0, zoom=2.0)

for psm in [3, 6, 11]:
    cfg = f"--psm {psm}"
    text = pytesseract.image_to_string(img_invoice, config=cfg)
    word_count = len(text.split())
    print(f"PSM {psm:2d}: {word_count:4d} words | preview: {text[:80]!r}")

PDF: ./data/example-invoice-microsoft-azure-PAYG.pdf | Page: 0
Image size: 1584x1224 px | Mode: RGB | Zoom: 2.0x
PSM  3:   90 words | preview: 'Bill to\n\n§8 Windows Azure\n\nCustomer PO No. United States\n7 Ati\nInvoice No. E050\n'
PSM  6:   88 words | preview: 'am Bill to\n_— |\nCustomer PO No. United States\nAti\nInvoice No. E050\nBilling Cycle'
PSM 11:  100 words | preview: 'Bill to\n\n§8 Windows Azure\n\n|\n\nUnited States\n\nCustomer PO No.\n\nAti\n\nInvoice No.\n\n'


In [21]:
# OCR Engine Mode (--oem N) selects the underlying Tesseract engine
# 0 — Legacy Tesseract only
# 1 — Neural net LSTM only
# 2 — Legacy + LSTM combined
# 3 — Default (best available, usually LSTM)
img_invoice = helper_pdf_page_to_image(files_list[0], page_number=0, zoom=2.0)

for oem in [0, 1, 3]:
    cfg = f"--oem {oem} --psm 3"
    try:
        text = pytesseract.image_to_string(img_invoice, config=cfg)
        print(f"OEM {oem}: {len(text.split())} words | preview: {text[:80]!r}")
    except Exception as e:
        print(f"OEM {oem}: error — {e}")

PDF: ./data/example-invoice-microsoft-azure-PAYG.pdf | Page: 0
Image size: 1584x1224 px | Mode: RGB | Zoom: 2.0x
OEM 0: error — (1, "Error: Tesseract (legacy) engine requested, but components are not present in /opt/homebrew/share/tessdata/eng.traineddata!! Failed loading language 'eng' Tesseract couldn't load any languages! Could not initialize tesseract.")
OEM 1: 90 words | preview: 'Bill to\n\n§8 Windows Azure\n\nCustomer PO No. United States\n7 Ati\nInvoice No. E050\n'
OEM 3: 90 words | preview: 'Bill to\n\n§8 Windows Azure\n\nCustomer PO No. United States\n7 Ati\nInvoice No. E050\n'


### Word-Level Data
Using `image_to_data()` to get per-word bounding boxes and confidence scores

In [22]:
# image_to_data with output_type=Output.DATAFRAME — far easier to analyse than raw TSV
# Columns: level, page_num, block_num, par_num, line_num, word_num,
#          left, top, width, height, conf, text
img = helper_pdf_page_to_image(files_list[1], page_number=0, zoom=2.0)

df_data = pytesseract.image_to_data(img, output_type=Output.DATAFRAME)

print(f"=== image_to_data as DataFrame: {len(df_data)} rows ===\n")
print(df_data.dtypes)
print()
display(df_data.head(12))

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x
=== image_to_data as DataFrame: 208 rows ===

level          int64
page_num       int64
block_num      int64
par_num        int64
line_num       int64
word_num       int64
left           int64
top            int64
width          int64
height         int64
conf         float64
text             str
dtype: object



,level,page_num,block_num,par_num,line_num,word_num,left,top,width,height,conf,text
0,1,1,0,0,0,0,0,0,449,1337,-1.000000,NaN
1,2,1,1,0,0,0,114,108,208,40,-1.000000,NaN
2,3,1,1,1,0,0,114,108,208,40,-1.000000,NaN
3,4,1,1,1,1,0,114,108,208,17,-1.000000,NaN
4,5,1,1,1,1,1,114,109,26,16,96.247269,THE
5,5,1,1,1,1,2,151,108,94,17,96.597801,CHEESECAKE
6,5,1,1,1,1,3,257,108,65,17,96.711937,FACTORY
7,4,1,1,1,2,0,189,131,65,17,-1.000000,NaN
8,5,1,1,1,2,1,189,131,65,17,96.473450,SEATTLE
9,2,1,2,0,0,0,26,178,344,64,-1.000000,NaN


In [23]:
# Filter out low-confidence entries (conf=-1 are non-word layout blocks)
df_words = df_data[(df_data['conf'] > 0)].copy()

print(f"Total entries:             {len(df_data)}")
print(f"Words with confidence > 0: {len(df_words)}")
print(f"Confidence range: {df_words['conf'].min():.0f} – {df_words['conf'].max():.0f}")
print(f"Mean confidence:  {df_words['conf'].mean():.1f}\n")

# High-confidence words (conf >= 80)
high_conf = df_words[df_words['conf'] >= 80]
print(f"High-confidence words (conf >= 80): {len(high_conf)}")
display(high_conf[['text', 'conf', 'left', 'top', 'width', 'height']].head(10))

Total entries:             208
Words with confidence > 0: 140
Confidence range: 14 – 97
Mean confidence:  84.1

High-confidence words (conf >= 80): 111


,text,conf,left,top,width,height
4,THE,96.247269,114,109,26,16
5,CHEESECAKE,96.597801,151,108,94,17
6,FACTORY,96.711937,257,108,65,17
8,SEATTLE,96.473450,189,131,65,17
12,O235,86.927582,27,178,73,17
13,TABLE,95.237007,122,178,47,17
14,203,91.356689,180,178,27,16
15,#Party,91.356689,227,178,56,19
16,1,96.971619,298,178,7,17
18,DALTON,81.805504,27,202,56,16


### Batch Processing
Running OCR across all documents and collecting results into a summary DataFrame

In [24]:
# Process all PDF files and collect OCR summary into a DataFrame
results = []

for file_path in tqdm(files_list):
    try:
        img = helper_pdf_page_to_image(file_path, page_number=0, zoom=2.0)
        text = pytesseract.image_to_string(img)
        df_page = pytesseract.image_to_data(img, output_type=Output.DATAFRAME)
        df_conf = df_page[df_page['conf'] > 0]

        results.append({
            'file': Path(file_path).name,
            'word_count': len(text.split()),
            'char_count': len(text),
            'mean_confidence': round(df_conf['conf'].mean(), 1) if len(df_conf) else 0,
        })
    except Exception as e:
        results.append({'file': Path(file_path).name, 'error': str(e)})

df_results = pd.DataFrame(results)
display(df_results)

  0%|                                                                                                           | 0/8 [00:00<?, ?it/s]

PDF: ./data/example-invoice-microsoft-azure-PAYG.pdf | Page: 0
Image size: 1584x1224 px | Mode: RGB | Zoom: 2.0x


 12%|████████████▍                                                                                      | 1/8 [00:00<00:04,  1.55it/s]

PDF: ./data/example-receipts-kaggle-cheesecakefactory.pdf | Page: 0
Image size: 449x1337 px | Mode: RGB | Zoom: 2.0x


 25%|████████████████████████▊                                                                          | 2/8 [00:01<00:03,  1.57it/s]

...

PDF: ./data/example-receipts-kaggle-wholefoods_1.pdf | Page: 0
Image size: 448x995 px | Mode: RGB | Zoom: 2.0x


 88%|██████████████████████████████████████████████████████████████████████████████████████▋            | 7/8 [00:04<00:00,  1.62it/s]

PDF: ./data/example-receipts-kaggle-wholefoods_2.pdf | Page: 0
Image size: 447x1119 px | Mode: RGB | Zoom: 2.0x


100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.53it/s]


,file,word_count,char_count,mean_confidence
0,example-invoice-microsoft-azure-PAYG.pdf,90,633,88.3
1,example-receipts-kaggle-cheesecakefactory.pdf,142,790,84.1
2,example-receipts-kaggle-luckylouie.pdf,104,751,86.2
3,example-receipts-kaggle-mcdonald.pdf,148,963,87.3
4,example-receipts-kaggle-oldnavy.pdf,257,1588,73.9
5,example-receipts-kaggle-spaceneedle.pdf,36,250,78.0
6,example-receipts-kaggle-wholefoods_1.pdf,154,960,84.1
7,example-receipts-kaggle-wholefoods_2.pdf,164,1042,84.7
